##### First PipeLine in RAG 

In [2]:
# ============================================================
# NAIVE RAG PIPELINE
# PDF → Chunks → Embeddings → Vector DB → Retrieval → LLM
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


# ============================================================
# 2. LOAD ENVIRONMENT VARIABLES
# ============================================================

# Loads API keys from the local .env file
load_dotenv()


# ============================================================
# PIPELINE 1: INDEXING / INGESTION PIPELINE
# ============================================================


# ============================================================
# 3. LOAD THE PDF DOCUMENT
# ============================================================

loader = PyPDFLoader("../data/Transformer.pdf")

# Convert PDF pages into LangChain Document objects
documents = loader.load()

print(f"Loaded {len(documents)} pages")


# ============================================================
# 4. SPLIT DOCUMENT INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

# Split large documents into smaller chunks
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")


# ============================================================
# 5. CREATE EMBEDDINGS
# ============================================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)


# ============================================================
# 6. STORE EMBEDDINGS IN CHROMA VECTOR DATABASE
# ============================================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print("Vector database created successfully!")


# ============================================================
# PIPELINE 2: RETRIEVAL PIPELINE
# ============================================================


# ============================================================
# 7. RETRIEVE RELEVANT CONTEXT
# ============================================================

def get_context(query: str):
    """
    Searches the vector database and retrieves the most relevant
    document chunks for the user's question.
    """

    # Retrieve the top 4 most relevant chunks
    retrieved_documents = vector_store.similarity_search(
        query,
        k=4,
    )

    # Combine the retrieved chunks into one context string
    context = "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )

    # Return data required by the prompt
    return {
        "context": context,
        "query": query,
    }


# ============================================================
# PIPELINE 3: AUGMENTATION + GENERATION PIPELINE
# ============================================================


# ============================================================
# 8. CREATE THE PROMPT TEMPLATE
# ============================================================

prompt = PromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I don't have enough information in the provided document."

Context:
{context}

Question:
{query}

Answer:
"""
)


# ============================================================
# 9. INITIALIZE THE LANGUAGE MODEL
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.2,
)


# ============================================================
# 10. CREATE THE RAG CHAIN
# ============================================================

rag_chain = get_context | prompt | llm


# ============================================================
# 11. ASK A QUESTION
# ============================================================

query = "Why was the Transformer needed?"

response = rag_chain.invoke(query)


# ============================================================
# 12. PRINT THE FINAL ANSWER
# ============================================================

# Gemini may return content as a list of content blocks
print(response.content[0]["text"])

C:\Users\HP\AppData\Local\Temp\ipykernel_19220\4153451137.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 14 pages
Created 21 chunks
Vector database created successfully!
Based on the provided context, the Transformer was needed to address the limitations of previous sequence-processing architectures like RNNs, LSTMs, and GRUs. Specifically, it solved two major problems:

1. **Sequential Processing:** Older models processed words one by one, making it impossible to efficiently process all words in parallel.
2. **Long-Term Dependencies:** Older models struggled to remember relationships between words that were far apart in a sequence.

By using Self-Attention, the Transformer allows every word to look at every other word simultaneously. This makes models:
* Highly parallelizable
* Better at understanding context
* Effective for long-range relationships
* Suitable for large-scale training
